## **Inventory Optimization Dashboard** — Data Inspection & Preparation

This notebook covers the initial inspection, data quality checks, cleaning
and preparation of the supply chain inventory dataset for further analysis
and Power BI dashboard development.

#### Dataset
- Records: 91,250
- SKUs: 50
- Warehouses: 5
- Suppliers: 10
- Regions: 4
- Date range: January 1, 2024 to December 30, 2024

## A. Load Dataset

#### 1. Import Libraries

In [ ]:
import pandas as pd
import os

#### 2. Loading raw Data

In [ ]:


df = pd.read_csv("../Data/supply_chain_dataset1.csv")

print("Shape:", df.shape)

#Initial inspection
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

#Missing values check
print("\nMissing values:")
print(df.isnull().sum())

#### 3. Statistical Summary

In [ ]:
df.describe(include="all").T

## B. Data Quality Checks

#### 1. Duplicate Check

In [ ]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)



#### 2. Date Validation

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

print(df["Date"].dtype)
print("Invalid dates:", df["Date"].isna().sum())

#### 3. Numerical Validation

In [ ]:
numeric_columns = [
    "Units_Sold",
    "Inventory_Level",
    "Supplier_Lead_Time_Days",
    "Reorder_Point",
    "Order_Quantity",
    "Unit_Cost",
    "Unit_Price",
    "Promotion_Flag",
    "Stockout_Flag",
    "Demand_Forecast"
]

df[numeric_columns].describe().T

#missing value validation
print("Total missing values:", df.isnull().sum().sum())

#### 4. Dataset Behaviour Checks

In [ ]:
print("Stockout flag values:")
print(df["Stockout_Flag"].value_counts(dropna=False))

print("\nOrder quantity > 0:", (df["Order_Quantity"] > 0).sum())
print("Order quantity = 0:", (df["Order_Quantity"] == 0).sum())

print("\nInventory at or below reorder point:")
print((df["Inventory_Level"] <= df["Reorder_Point"]).sum())

In [ ]:
print("Unique SKUs:", df["SKU_ID"].nunique())
print("Unique Warehouses:", df["Warehouse_ID"].nunique())
print("Unique Suppliers:", df["Supplier_ID"].nunique())
print("Unique Regions:", df["Region"].nunique())

print("\nDate range:")
print(df["Date"].min(), "to", df["Date"].max())

## C. Data Preparation

#### 1.

In [ ]:
# Ensure Date is properly formatted
df["Date"] = pd.to_datetime(df["Date"])

# Convert categorical columns to category type
categorical_columns = [
    "SKU_ID",
    "Warehouse_ID",
    "Supplier_ID",
    "Region"
]

for col in categorical_columns:
    df[col] = df[col].astype("category")



#### 2. Feature Engineering

In [ ]:
# Create inventory value
df["Inventory_Value"] = (
    df["Inventory_Level"] * df["Unit_Cost"]
)

# Create unit margin
df["Unit_Margin"] = (
    df["Unit_Price"] - df["Unit_Cost"]
)

# Create demand variance
df["Demand_Variance"] = (
    df["Units_Sold"] - df["Demand_Forecast"]
)

# Create reorder risk indicator
df["Reorder_Risk"] = (
    df["Inventory_Level"] <= df["Reorder_Point"]
).astype(int)

print("Prepared shape:", df.shape)

print("\nNew columns:")
print([
    "Inventory_Value",
    "Unit_Margin",
    "Demand_Variance",
    "Reorder_Risk"
])

In [ ]:
print(df[
    [
        "Inventory_Value",
        "Unit_Margin",
        "Demand_Variance",
        "Reorder_Risk"
    ]
].describe())

#### 3. Feature Validation

In [ ]:
print("\nReorder Risk distributions:")
print(df["Reorder_Risk"].value_counts())

print("\nPrepared data shapes:")
print(df.shapes)

### D. Saving cleaned Dataset

In [ ]:

file_path = "../Data/inventory_cleaned.csv"
df.to_csv(file_path, index=False)

print("Cleaned dataset saved successfully.")
print("File exists:", os.path.exists(file_path))